In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer,BitsAndBytesConfig

In [2]:
import transformers
transformers.__version__

'4.46.3'

In [3]:
try:
    del model
except :
    pass

try:    
    del tokenizer
except:
    pass

import gc
gc.collect()

import torch
torch.cuda.empty_cache()

In [4]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer,BitsAndBytesConfig
model_id = 'Qwen/Qwen2.5-0.5B-Instruct'
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16    
)
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb_config, device_map='auto')

In [5]:
from datasets import load_dataset
# 범용 비즈니스 및 지시어 데이터셋 (KoAlpaca) 로드
dataset = load_dataset("beomi/KoAlpaca-v1.1a", split="train")
# 모델이 알아들을 수 있도록 Qwen Chat Template 적용 함수
def format_prompt(example):
    messages = [
        {"role": "system", "content": "You are a helpful AI assistant. Respond in Korean."},
        {"role": "user", "content": example["instruction"]}
    ]
    
    # 1. 챗 템플릿 적용
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    
    # 2.  만약 버그로 인해 리스트(숫자)가 반환되었다면 강제로 문자열로 디코딩
    if isinstance(text, list):
        text = tokenizer.decode(text)
        
    # 3. 정답(output)과 종료 토큰 추가
    text += example["output"] + tokenizer.eos_token
    return {"text": text}
# 실습 시간 단축을 위해 2,000건의 데이터만 샘플링하여 훈련 세트로 구성
# train_dataset = dataset.select(range(2000)).map(format_prompt)
train_dataset = dataset.map(format_prompt)
print("\n[전처리된 데이터 예시]")
print(train_dataset[0]["text"])


[전처리된 데이터 예시]
<|im_start|>system
You are a helpful AI assistant. Respond in Korean.<|im_end|>
<|im_start|>user
양파는 어떤 식물 부위인가요? 그리고 고구마는 뿌리인가요?<|im_end|>
<|im_start|>assistant
양파는 잎이 아닌 식물의 줄기 부분입니다. 고구마는 식물의 뿌리 부분입니다. 

식물의 부위의 구분에 대해 궁금해하는 분이라면 분명 이 질문에 대한 답을 찾고 있을 것입니다. 양파는 잎이 아닌 줄기 부분입니다. 고구마는 다른 질문과 답변에서 언급된 것과 같이 뿌리 부분입니다. 따라서, 양파는 식물의 줄기 부분이 되고, 고구마는 식물의 뿌리 부분입니다.

 덧붙이는 답변: 고구마 줄기도 볶아먹을 수 있나요? 

고구마 줄기도 식용으로 볶아먹을 수 있습니다. 하지만 줄기 뿐만 아니라, 잎, 씨, 뿌리까지 모든 부위가 식용으로 활용되기도 합니다. 다만, 한국에서는 일반적으로 뿌리 부분인 고구마를 주로 먹습니다.<|im_end|>


In [6]:
train_dataset

Dataset({
    features: ['instruction', 'output', 'url', 'text'],
    num_rows: 21155
})

In [7]:
dataset.select(range(2000))['instruction']

Column(['양파는 어떤 식물 부위인가요? 그리고 고구마는 뿌리인가요?', '스웨터의 유래는 어디에서 시작되었나요?', '토성의 고리가 빛의 띠로 보이는 이유는 무엇인가요?  \n\n토성의 고리는 얼음과 같은 여러 물질로 이루어져 있다고 알고 있는데, 카시니가 찍은 사진에서 마치 빛의 띠 처럼 보이는 이유가 무엇인가요? 물질의 공전 속도가 빠르기 때문에 카메라로 담았을 때 빛의 궤적으로 보이는 건가요? 또한, 야간에 빠르게 움직이는 자동차를 장노출로 찍었을 때 빛의 궤적이 생기는 것과 같은 원리일까요? 그리고 빛의 궤적이 생기는 것은 우주라는 어두운 환경 특성 때문이라고 생각됩니다. 이게 맞을까요?', '화장품 OEM과 화장품 ODM의 차이점은 무엇인가요?\n화장품 자체 제조 브랜드 런칭을 위해 OEM과 ODM용어에 대해 혼란스러움을 느끼고 있습니다. 두 용어의 차이점이 무엇인지 알고 싶습니다.', "'사이보그'는 언제 처음 등장한 말이며, 그 의미와 종류에는 어떤 것이 있는지 알고 싶습니다.", ...])

In [8]:
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

# k-bit 양자화된 모델을 파인튜닝할 수 있도록 전처리
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# 최신 trl 라이브러리에 맞추어 TrainingArguments 대신 SFTConfig 사용
training_args = SFTConfig(
    output_dir="./qwen-koalpaca-lora",
    per_device_train_batch_size=100,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    # max_steps=100, #  100 step만 진행    
    num_train_epochs=10,
    optim="paged_adamw_8bit",
    fp16=False,
    bf16=True, # H100의 BFloat16 가속 활성화
    dataset_text_field="text", 
    max_seq_length=512,        
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    args=training_args,
    peft_config=lora_config
)

In [9]:
# 모델 파인튜닝
trainer.train()

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice:  3


wandb: You chose "Don't visualize my results"
wandb: Using W&B in offline mode.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...
/usr/local/lib/python3.11/dist-packages/torch/_dynamo/eval_frame.py:600: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.4 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


OutOfMemoryError: CUDA out of memory. Tried to allocate 28.92 GiB. GPU 0 has a total capacity of 79.18 GiB of which 260.19 MiB is free. Including non-PyTorch memory, this process has 78.92 GiB memory in use. Of the allocated memory 77.28 GiB is allocated by PyTorch, and 1000.66 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
# 모델 병합
from peft import PeftModel

# del model
# del tainer
# torch.cuda.empty_cache()
# 원본모델을 양자화 없이 FP16으로 로드
base_model =  AutoModelForCausalLM.from_pretrained(
    model_id,
    low_cpu_mem_usage=True,
    return_dict=True,
    torch_dtype=torch.float16,
    device_map='auto'
)
# 방금 학습한 LoRA 가중치를 불러와서 update
model = PeftModel.from_pretrained(base_model, './qwen-koalpaca-lora/checkpoint-100')
# 가중치 최종 병합
merged_model = model.merge_and_unload()
# 병합된 모델을 저장
save_directory = "./Qwen-KoAlpaca-Merged"
merged_model.save_pretrained(save_directory)

In [ ]:
save_directory